In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error
import re
import csv

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        holdout = candidate / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_ML' / 'heloc_ML_holdout.csv'
        if holdout.exists():
            return candidate
    raise FileNotFoundError('Impossibile trovare la radice del progetto.')

project_root = find_project_root()
clean_test_path = project_root / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_ML' / 'heloc_ML_imputation_test.csv'
mask_dir = project_root / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
imputated_root = project_root / 'data' / 'processed' / 'Fase3' / 'Imputated_ML'
output_dir = project_root / 'data' / 'processed' / 'Fase3' / 'Results'

print(f'Project root : {project_root}')
print(f'Clean Test   : {clean_test_path.exists()}')
print(f'Mask Dir     : {mask_dir.exists()}')
print(f'Imputated Dir: {imputated_root.exists()}')
print(f'Output Dir   : {output_dir}')

Project root : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project
Clean Test   : True
Mask Dir     : True
Imputated Dir: True
Output Dir   : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Results


In [ ]:
# 1. Carichiamo il dataset pulito (Ground Truth)
df_clean = pd.read_csv(clean_test_path)

# La variabile target (RiskPerformance) non viene corrotta né imputata, quindi la ignoriamo.
if 'RiskPerformance' in df_clean.columns:
    df_clean = df_clean.drop(columns=['RiskPerformance'])

results = []
imputation_methods = ['Mediana', 'MICE']

for method in imputation_methods:
    method_dir = imputated_root / method
    imputed_files = sorted(method_dir.glob('*_discrirminative_train_*.csv'))
    
    for imp_path in imputed_files:
        # Estrarre strategia e percentuale dal nome file
        m = re.match(r'^(.+?)_discrirminative_train_([A-Z]+)_(\d+)\.csv$', imp_path.name)
        if not m:
            continue
        dataset, strategy, pct = m.groups()
        
        # 2. Caricare il dataset imputato
        df_imputed = pd.read_csv(imp_path)
        if 'RiskPerformance' in df_imputed.columns:
            df_imputed = df_imputed.drop(columns=['RiskPerformance'])
            
        # 3. Caricare la corrispondente maschera di valori mancanti (True = valore mancante)
        mask_name = f'{dataset}_imputation_test_mask_{strategy}_{pct}.csv'
        mask_path = mask_dir / mask_name
        df_mask = pd.read_csv(mask_path)
        if 'RiskPerformance' in df_mask.columns:
            df_mask = df_mask.drop(columns=['RiskPerformance'])
            
        # Assicuriamoci che i dataset abbiano la stessa forma
        assert df_clean.shape == df_imputed.shape == df_mask.shape
        
        # Convertiamo a numpy arrays per velocità
        clean_vals = df_clean.values
        imputed_vals = df_imputed.values
        mask_vals = df_mask.values
        
        # 4. Calcoliamo MSE/MAE esclusivamente sui valori mancanti
        clean_missing_only = clean_vals[mask_vals]
        imputed_missing_only = imputed_vals[mask_vals]
        
        mse = mean_squared_error(clean_missing_only, imputed_missing_only)
        mae = mean_absolute_error(clean_missing_only, imputed_missing_only)
        
        results.append({
            'imputation_method': method,
            'dataset': dataset,
            'missing_strategy': strategy,
            'missing_pct': int(pct),
            'mse': mse,
            'mae': mae
        })
        print(f'{method} | {strategy} {pct}% -> MSE: {mse:.4f}, MAE: {mae:.4f}')

Mediana | MAR 10% -> MSE: 179.0524, MAE: 2.1404
Mediana | MAR 25% -> MSE: 104.8731, MAE: 4.0521
Mediana | MAR 40% -> MSE: 635.6480, MAE: 7.4715
Mediana | MCAR 10% -> MSE: 35.4138, MAE: 1.3327
Mediana | MCAR 25% -> MSE: 110.4785, MAE: 3.7374
Mediana | MCAR 40% -> MSE: 765.5818, MAE: 11.1169
Mediana | MNAR 10% -> MSE: 178.3672, MAE: 2.8291
Mediana | MNAR 25% -> MSE: 105.5662, MAE: 4.0610
Mediana | MNAR 40% -> MSE: 133.1507, MAE: 4.9578
MICE | MAR 10% -> MSE: 77.3898, MAE: 1.3914
MICE | MAR 25% -> MSE: 52.2955, MAE: 2.7539
MICE | MAR 40% -> MSE: 486.8055, MAE: 6.2673
MICE | MCAR 10% -> MSE: 19.5445, MAE: 0.8926
MICE | MCAR 25% -> MSE: 52.7116, MAE: 2.4393
MICE | MCAR 40% -> MSE: 553.7517, MAE: 9.0452
MICE | MNAR 10% -> MSE: 86.4450, MAE: 1.7629
MICE | MNAR 25% -> MSE: 57.4136, MAE: 2.8749
MICE | MNAR 40% -> MSE: 74.0813, MAE: 3.7665


In [3]:
# 5. Salviamo i risultati
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / 'baseline_imputation_mse_mae.csv'

df_results = pd.DataFrame(results)
df_results.to_csv(report_path, index=False)

print(f'\nRisultati di Baseline salvati in: {report_path}')

# Mostriamo la tabella finale
df_results.sort_values(by=['imputation_method', 'missing_strategy', 'missing_pct'])


Risultati di Baseline salvati in: /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Results/baseline_imputation_mse_mae.csv


,imputation_method,dataset,missing_strategy,missing_pct,mse,mae
9,MICE,heloc_ML,MAR,10,77.389833,1.391432
10,MICE,heloc_ML,MAR,25,52.295490,2.753907
11,MICE,heloc_ML,MAR,40,486.805474,6.267326
12,MICE,heloc_ML,MCAR,10,19.544473,0.892591
13,MICE,heloc_ML,MCAR,25,52.711602,2.439348
14,MICE,heloc_ML,MCAR,40,553.751695,9.045210
15,MICE,heloc_ML,MNAR,10,86.444999,1.762896
16,MICE,heloc_ML,MNAR,25,57.413650,2.874908
17,MICE,heloc_ML,MNAR,40,74.081345,3.766491
0,Mediana,heloc_ML,MAR,10,179.052431,2.140418
